# 03 — Semantic Search Demo
Build sentence embeddings for every job posting, index them with FAISS, and retrieve the Top-K jobs most similar to a sample candidate profile.

> If `sentence-transformers` model weights can't be downloaded (e.g. no internet access), `EmbeddingEngine` automatically falls back to TF-IDF — the rest of this notebook works either way.

In [13]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
from src.utils import load_jobs_dataframe
from src.embedding_engine import create_embeddings
from src.vector_search import build_faiss_index, search_similar_jobs

jobs_df = load_jobs_dataframe()
corpus = (jobs_df['job_title'] + ' ' + jobs_df['description'] + ' ' + jobs_df['skills']).tolist()
vectors, engine = create_embeddings(corpus)
print('Embedding backend in use:', engine.backend_name)
print('Embedding matrix shape:', vectors.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2568.68it/s]
c:\Users\Administrator\Downloads\Projects\PathForge\src\embedding_engine.py:61: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = self._model.get_sentence_embedding_dimension()
[INFO] pathforge.embedding_engine: Loaded SentenceTransformer backend: sentence-transformers (all-MiniLM-L6-v2)


Embedding backend in use: sentence-transformers (all-MiniLM-L6-v2)
Embedding matrix shape: (80, 384)


## Build the FAISS (or fallback) index

In [14]:
index = build_faiss_index(vectors, jobs_df['job_id'].tolist())
print('Index backend:', index.backend_name)

[INFO] pathforge.vector_search: Built faiss index over 80 job postings.


Index backend: faiss


## Query with a sample candidate profile

In [15]:
query = 'Skilled in Python, machine learning, pandas, scikit-learn, and SQL for data analysis.'
query_vec = engine.encode([query])[0]
results = search_similar_jobs(index, query_vec, top_k=5)
for job_id, score in results:
    row = jobs_df[jobs_df.job_id == job_id].iloc[0]
    print(f'{score:.3f}  {row.job_title:28s} @ {row.company}')

0.763  Data Scientist               @ Northwind AI
0.720  Data Scientist               @ Corestack Solutions
0.720  Data Scientist               @ Vertex Data Labs
0.699  Data Analyst                 @ Bluepeak Software
0.699  Data Scientist               @ Skyline Systems


## Inspect the top match in full

In [16]:
top_job_id, top_score = results[0]
top_row = jobs_df[jobs_df.job_id == top_job_id].iloc[0]
print(top_row['description'])
print('\nRequired skills:', top_row['skills'])

We are looking for a entry-level Data Scientist to join our finance team. The ideal candidate has hands-on experience with sql, numpy, scikit-learn, pandas, python, machine learning, deep learning, data analysis. You will work on real-world problems, collaborate with cross-functional teams, and help ship production-quality solutions.

Required skills: sql; numpy; scikit-learn; pandas; python; machine learning; deep learning; data analysis
